In [103]:
import torch
import numpy as np

from utils import *

In [104]:
d1 = 10000
d2 = 1000
r = 10
p = 0.01

In [105]:
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
print(dataset)

M = load_data_syn(r, d1, d2, device)

# sample 2 entries per row
#observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), 2)
#observed_M = torch.from_numpy(observed_M).float().to(device)
#masks = torch.from_numpy(masks).to(device)

# IID sample
observed_M, masks = get_uniform_masks(M, p)

# observed second-moment matrix
MTM = M.T @ M
cov_observe_M =  observed_M.T @ observed_M
T_masks = 1 * (cov_observe_M!=0)

syn


In [106]:
# True probability weighting
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_p:  tensor(1.0076, device='cuda:0')


In [107]:
# single value estimated probability weighting
p_hat = (observed_M!=0).sum()/(d1*d2)
T_single_p_hat = ((1.0 / p_hat) * diag_cov + (1.0 / (p_hat**2)) * (cov_observe_M - diag_cov))
print("relative error of T_single_p_hat: ", torch.norm(T_single_p_hat*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_single_p_hat:  tensor(1.0065, device='cuda:0')


In [108]:
print(p)
print(p_hat)

0.01
tensor(0.0100, device='cuda:0')


In [109]:
# Inverse estimated probability weighting in matrix
cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
p_hat_matrix = cov_observe_count/d1

T = cov_observe_M / p_hat_matrix

# calculate matrix of p_hat and p
p_hat_matrix = (cov_observe_count/d1)
p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)

# calculate (p_hat - p)
print("mean value of (p_hat - p) in matrix: ", (p_matrix*T_masks-p_hat_matrix*T_masks).mean()/(T_masks.sum() / d2**2))
# calculate the relative err of T_p_hat
print("relative error of T_p_hat: ", torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))

mean value of (p_hat - p) in matrix:  tensor(-5.8312e-05, device='cuda:0')
relative error of T_p_hat:  tensor(0.0813, device='cuda:0')


In [110]:
print(p_hat_matrix)

tensor([[1.0700e-02, 4.0000e-04, 3.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         2.0000e-04],
        [4.0000e-04, 9.3000e-03, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [3.0000e-04, 1.0000e-04, 7.5000e-03,  ..., 1.0000e-04, 1.0000e-04,
         2.0000e-04],
        ...,
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 8.9000e-03, 2.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 2.0000e-04, 9.7000e-03,
         1.0000e-04],
        [2.0000e-04, 1.0000e-04, 2.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         9.3000e-03]], device='cuda:0')


In [111]:
print(p_matrix)

tensor([[1.0000e-02, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-02, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-02,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        ...,
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-02, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-02,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-02]], device='cuda:0')


In [112]:
# calculate Taylor expansion in diagonal elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )

T_p_1_diag = (-1.0 / p**2) * diag_cov
T_p_2_diag = (1.0 / p**3) * diag_cov 
T_p_3_diag = (-1.0 / p**4) * diag_cov 

item_1 = torch.mul(T_p_1_diag,(p_hat_matrix-p_matrix).diag())
item_2 = torch.mul(T_p_2_diag,(p_hat_matrix-p_matrix).diag()**2)
item_3 = torch.mul(T_p_3_diag,(p_hat_matrix-p_matrix).diag()**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item in diag: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item in diag: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item in diag: ", (item_3*T_masks).sum()/num)

value of 1st-order item in diag:  tensor(-0.6099, device='cuda:0')
value of 2nd-order item in diag:  tensor(0.5906, device='cuda:0')
value of 2th-order item in diag:  tensor(-0.0185, device='cuda:0')


In [113]:
# calculate Taylor expansion in off-diag elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p_1_offdiag = - (2.0 / (p**3)) * (cov_observe_M - diag_cov)
T_p_2_offdiag = (3.0 / (p**4)) * (cov_observe_M - diag_cov)
T_p_3_offdiag =  - (4.0 / (p**5)) * (cov_observe_M - diag_cov)

delta_p_offdiag = (p_hat_matrix-p_matrix).fill_diagonal_(0)
item_1 = torch.mul(T_p_1_offdiag,delta_p_offdiag)
item_2 = torch.mul(T_p_2_offdiag,delta_p_offdiag**2)
item_3 = torch.mul(T_p_3_offdiag,delta_p_offdiag**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = T_masks.sum() - (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item: ", (item_3*T_masks).sum()/num)

value of 1st-order item:  tensor(-1268.2295, device='cuda:0')
value of 2nd-order item:  tensor(37.9406, device='cuda:0')
value of 2th-order item:  tensor(-1.2584, device='cuda:0')


In [ ]:
import torch
import numpy as np

from utils import *

def normalize_matrix(matrix):
    """
    Normalize a PyTorch tensor matrix:
    - Diagonal elements are divided by the sum of squares of the diagonal elements.
    - Off-diagonal elements are divided by the sum of squares of the off-diagonal elements.
    
    Args:
        matrix (torch.Tensor): Input square matrix.

    Returns:
        torch.Tensor: Normalized matrix.
    """
    if not matrix.ndim == 2 or matrix.size(0) != matrix.size(1):
        raise ValueError("Input must be a square matrix")
    
    # Get the diagonal and off-diagonal masks
    diagonal_mask = torch.eye(matrix.size(0), dtype=torch.bool, device=matrix.device)
    off_diagonal_mask = ~diagonal_mask
    
    # Extract diagonal and off-diagonal elements
    diagonal_elements = matrix[diagonal_mask]
    off_diagonal_elements = matrix[off_diagonal_mask]
    
    # Calculate sums of squares
    diagonal_square_sum = torch.sqrt(torch.sum(diagonal_elements ** 2))
    off_diagonal_square_sum = torch.sqrt(torch.sum(off_diagonal_elements ** 2))
    
    # Normalize
    normalized_matrix = torch.zeros_like(matrix)
    normalized_matrix[diagonal_mask] = diagonal_elements / diagonal_square_sum
    normalized_matrix[off_diagonal_mask] = off_diagonal_elements / off_diagonal_square_sum
    
    return normalized_matrix

d1 = 10000
d2 = 250
r = 10
p = 0.1

if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
#print(dataset)

total_t = 5
total_T_var_list = []
total_T_p_var_list = []
total_T_err_list = []
total_T_p_err_list = []
for t in range(total_t):
    M = load_data_syn(r, d1, d2, device)
    
    runs = 100
    T_p_list = []
    T_list = []
    T_p_err_list = []
    T_err_list = []
    for run in range(runs):
        
        #M = torch.ones(d1,d2).to(device)
        #M = torch.normal(2, 1, size = (d1, d2)).to(device)

        # sample 2 entries per row
        #observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), 2)
        #p=2/d2
        #observed_M = torch.from_numpy(observed_M).float().to(device)
        #masks = torch.from_numpy(masks).to(device)

        # IID sample
        #observed_M, masks = get_uniform_masks(M, p)

        # few entry sample
        ob = 10
        observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
        p = ob/d2
        observed_M = torch.from_numpy(observed_M).float().to(device)
        masks = torch.from_numpy(masks).to(device)

        # observed second-moment matrix
        MTM = M.T @ M
        cov_observe_M =  observed_M.T @ observed_M
        T_masks = 1 * (cov_observe_M!=0)

        # True probability weighting
        diag_cov = torch.diag( torch.diag(cov_observe_M) )
        T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
        #T_p = normalize_matrix(T_p)
        T_p_err_list.append((torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks)).item())
        T_p = T_p / torch.mean(T_p)
        T_p_list.append(T_p.unsqueeze(0))
        #print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

        # Inverse estimated probability weighting in matrix
        cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
        cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
        p_hat_matrix = cov_observe_count/d1

        T = cov_observe_M / p_hat_matrix
        T_err_list.append((torch.norm(T*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks)).item())
        #T = normalize_matrix(T)
        T = T / torch.mean(T)
        T_list.append(T.unsqueeze(0))

        # calculate matrix of p_hat and p
        p_hat_matrix = (cov_observe_count/d1)
        p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)

        # calculate (p_hat - p)
        #print("mean value of (p_hat - p) in matrix: ", (p_matrix*T_masks-p_hat_matrix*T_masks).mean()/(T_masks.sum() / d2**2))
        # calculate the relative err of T_p_hat
        #print("relative error of T_p_hat: ", torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))
    #print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))
    #print("relative error of T: ", torch.norm(T*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))
    total_T_err_list.append(np.mean(T_err_list))
    total_T_p_err_list.append(np.mean(T_p_err_list))

    mean_T_p = torch.cat(T_p_list).mean(0)
    mean_T = torch.cat(T_list).mean(0)
    var_T_p = torch.cat(T_p_list).var(0)
    var_T = torch.cat(T_list).var(0)

    var_T_p_value = (var_T_p).mean()
    var_T_value = (var_T).mean()

    diag_var_T_p = torch.diag(torch.diag(var_T_p))
    offdiag_var_T_p = var_T_p - diag_var_T_p
    diag_var_T = torch.diag(torch.diag(var_T))
    offdiag_var_T = var_T - diag_var_T

    num_diag = d2
    num_off = d2*(d2-1)/2

    total_T_p_var_list.append(var_T_p_value.item())
    total_T_var_list.append(var_T_value.item())

    #print("var of T_p: ", var_T_p_value)
    #print("var of T: ", var_T_value)

    #print("var of T_p in diag: ", diag_var_T_p.sum())
    #print("var of T in diag: ", diag_var_T.sum())

    #print("var of T_p in off-diag: ", offdiag_var_T_p.sum())
    #print("var of T in off-diag: ", offdiag_var_T.sum())

print(f"mean of err of T_p: {np.mean(total_T_p_err_list)} +- {np.std(total_T_p_err_list)} ")
print(f"mean of err of T: {np.mean(total_T_err_list)} +- {np.std(total_T_err_list)} ")

print(f"mean of var of T_p: {np.mean(total_T_p_var_list)} +- {np.std(total_T_p_var_list)} ")
print(f"mean of var of T: {np.mean(total_T_var_list)} +- {np.std(total_T_var_list)} ")
